# Dog Emotion Classification — Goals-Complete Notebook

This notebook implements the full project requirements for classifying dog emotions (`angry`, `happy`, `relaxed`, `sad`) using transfer learning with ResNet-18 and ResNet-50.

It follows a clear run order and keeps all logic inside this notebook (no module split), while preserving `resnet.py` unchanged for reference.

---

## 1) Section: Set Up Notebook Environment
Import required Python libraries, enable reproducibility settings, and verify package versions in code cells.

In [ ]:
# Colab setup (safe to run locally too)
import os
import sys
import subprocess

IS_COLAB = 'google.colab' in sys.modules
print(f"Running in Colab: {IS_COLAB}")

def _pip_install(packages):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + packages
    subprocess.check_call(cmd)

if IS_COLAB:
    print('Installing notebook dependencies for Colab...')
    _pip_install([
        'torch',
        'torchvision',
        'scikit-learn',
        'seaborn',
        'kagglehub',
        'wandb',
    ])
    # Colab is often more stable with single-worker DataLoaders.
    os.environ.setdefault('FORCE_NUM_WORKERS', '0')
    print('Colab setup complete.')
else:
    print('Local runtime detected; skipping Colab package install.')

In [ ]:
import os
import random
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

try:
    import kagglehub
    HAS_KAGGLEHUB = True
except Exception:
    HAS_KAGGLEHUB = False


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Torch: {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"Numpy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2) Section: Configure Project Constants and Paths
Define configuration variables, file paths, and runtime parameters as editable constants for quick iteration.

In [ ]:
# Dataset path options:
# 1) If DATASET_ROOT points to a valid folder with class subfolders, notebook uses it.
# 2) Otherwise, if kagglehub is available, notebook downloads and uses the Kaggle dataset.
# Expected folder structure:
# dataset_root/
#   angry/
#   happy/
#   relaxed/
#   sad/

DATASET_ROOT = ''  # Optional local override, e.g. '/absolute/path/to/Dog Emotion'
OUTPUT_DIR = Path('outputs')
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FIGURE_DIR = OUTPUT_DIR / 'figures'

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = int(os.getenv('FORCE_NUM_WORKERS', '2'))
PIN_MEMORY = torch.cuda.is_available()

EPOCHS_FROZEN = 25
EPOCHS_FINETUNE = 15
LR_FROZEN = 1e-4
LR_FINETUNE = 1e-5
WEIGHT_DECAY = 1e-4
DROPOUT = 0.5

# Run-all defaults: full experiments; smoke demo disabled by default
RUN_FULL_EXPERIMENTS = True
RUN_SMOKE_DEMO = False
SMOKE_EPOCHS = 1

CLASS_NAMES = ['angry', 'happy', 'relaxed', 'sad']
NUM_CLASSES = len(CLASS_NAMES)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
print(f'Output directory: {OUTPUT_DIR.resolve()}')
print(f'NUM_WORKERS: {NUM_WORKERS}')
print(f'RUN_FULL_EXPERIMENTS: {RUN_FULL_EXPERIMENTS}')
print(f'RUN_SMOKE_DEMO: {RUN_SMOKE_DEMO}')

## 3) Section: Implement Core Data Structures
Create initial classes, dictionaries, or dataclasses that represent the main entities used by the implementation.

In [ ]:
@dataclass
class ExperimentConfig:
    model_name: str              # 'resnet18' or 'resnet50'
    fine_tuned: bool             # False=frozen backbone, True=fine-tune last blocks
    augmentation: bool           # training augmentation on/off
    frozen_epochs: int = EPOCHS_FROZEN
    finetune_epochs: int = EPOCHS_FINETUNE
    batch_size: int = BATCH_SIZE


@dataclass
class RunArtifacts:
    run_name: str
    model_name: str
    fine_tuned: bool
    augmentation: bool
    best_checkpoint: str
    accuracy: float
    precision_macro: float
    recall_macro: float
    f1_macro: float


results: List[Dict] = []
history_by_run: Dict[str, Dict[str, List[float]]] = {}
all_misclassified: Dict[str, pd.DataFrame] = {}

print('Data structures initialized.')

## 4) Section: Build Main Processing Function
Write the first working version of the core function, including input validation and clear return values.

This section provides the complete reusable pipeline functions used by all experiments.

In [ ]:
def resolve_dataset_root(local_root: str = DATASET_ROOT) -> Path:
    if local_root and Path(local_root).exists():
        path = Path(local_root)
        print(f'Using local dataset at: {path}')
        return path

    if not HAS_KAGGLEHUB:
        raise FileNotFoundError(
            'DATASET_ROOT not found and kagglehub unavailable. Set DATASET_ROOT to your dataset folder.'
        )

    downloaded = kagglehub.dataset_download('danielshanbalico/dog-emotion')
    candidate = Path(downloaded) / 'Dog Emotion'
    if not candidate.exists():
        raise FileNotFoundError(f'Expected folder not found at: {candidate}')
    print(f'Using Kaggle dataset at: {candidate}')
    return candidate


def build_transforms(use_augmentation: bool) -> transforms.Compose:
    if use_augmentation:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def stratified_indices_from_imagefolder(base_dataset: datasets.ImageFolder, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    labels = np.array(base_dataset.targets)
    all_idx = np.arange(len(labels))

    train_idx, temp_idx = train_test_split(
        all_idx,
        test_size=0.30,
        stratify=labels,
        random_state=seed,
    )

    temp_labels = labels[temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        stratify=temp_labels,
        random_state=seed,
    )

    return train_idx, val_idx, test_idx


def create_dataloaders(dataset_root: Path, augmentation: bool, batch_size: int = BATCH_SIZE) -> Tuple[Dict[str, DataLoader], List[str], Dict[str, np.ndarray]]:
    base_dataset = datasets.ImageFolder(root=dataset_root)
    class_names = base_dataset.classes

    if sorted(class_names) != sorted(CLASS_NAMES):
        print(f'Warning: class names differ from expected list. Found: {class_names}')

    train_idx, val_idx, test_idx = stratified_indices_from_imagefolder(base_dataset, seed=SEED)

    train_ds = datasets.ImageFolder(root=dataset_root, transform=build_transforms(use_augmentation=augmentation))
    eval_ds = datasets.ImageFolder(root=dataset_root, transform=build_transforms(use_augmentation=False))

    train_subset = Subset(train_ds, train_idx)
    val_subset = Subset(eval_ds, val_idx)
    test_subset = Subset(eval_ds, test_idx)

    loaders = {
        'train': DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        'val': DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
        'test': DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY),
    }

    split_indices = {
        'train': train_idx,
        'val': val_idx,
        'test': test_idx,
    }

    return loaders, class_names, split_indices


def create_model(model_name: str, num_classes: int = NUM_CLASSES, dropout: float = DROPOUT) -> nn.Module:
    name = model_name.lower()
    if name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = 512
    elif name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        in_features = 2048
    else:
        raise ValueError("model_name must be 'resnet18' or 'resnet50'")

    model.fc = nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(512, num_classes),
    )
    return model


def set_trainable_layers(model: nn.Module, fine_tuned: bool) -> nn.Module:
    for param in model.parameters():
        param.requires_grad = False

    for param in model.fc.parameters():
        param.requires_grad = True

    if fine_tuned:
        for param in model.layer4.parameters():
            param.requires_grad = True
        # Unfreeze layer3 as optional second block for deeper fine-tuning
        for param in model.layer3.parameters():
            param.requires_grad = True

    return model


def create_optimizer(model: nn.Module, fine_tuned: bool) -> optim.Optimizer:
    if fine_tuned:
        head_params = list(model.fc.parameters())
        backbone_params = [p for n, p in model.named_parameters() if ('layer3' in n or 'layer4' in n) and p.requires_grad]
        return optim.Adam([
            {'params': backbone_params, 'lr': LR_FINETUNE},
            {'params': head_params, 'lr': LR_FROZEN},
        ], weight_decay=WEIGHT_DECAY)

    params = [p for p in model.parameters() if p.requires_grad]
    return optim.Adam(params, lr=LR_FROZEN, weight_decay=WEIGHT_DECAY)


def run_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: Optional[optim.Optimizer] = None) -> Tuple[float, float]:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    epoch_loss = 0.0
    all_preds, all_labels = [], []

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            epoch_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = epoch_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc


def train_model(
    model: nn.Module,
    loaders: Dict[str, DataLoader],
    run_name: str,
    epochs: int,
    fine_tuned: bool,
) -> Tuple[nn.Module, Dict[str, List[float]], Path]:
    criterion = nn.CrossEntropyLoss()
    optimizer = create_optimizer(model, fine_tuned=fine_tuned)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    history = {
        'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []
    }

    best_val_acc = -1.0
    best_path = CHECKPOINT_DIR / f'{run_name}_best.pth'

    for epoch in range(epochs):
        train_loss, train_acc = run_epoch(model, loaders['train'], criterion, optimizer)
        val_loss, val_acc = run_epoch(model, loaders['val'], criterion, optimizer=None)
        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        print(
            f"[{run_name}] Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss {train_loss:.4f} | Train Acc {train_acc:.4f} | "
            f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f} | LR {current_lr:.2e}"
        )

    model.load_state_dict(torch.load(best_path, map_location=device))
    return model, history, best_path


def predict_with_probabilities(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = probs.argmax(dim=1)

            all_labels.extend(labels.numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str]) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))

    return {
        'accuracy': float(acc),
        'precision_macro': float(precision),
        'recall_macro': float(recall),
        'f1_macro': float(f1),
    }


def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str], title: str, save_name: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(title)
    out_path = FIGURE_DIR / save_name
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved confusion matrix: {out_path}')


def plot_history(history: Dict[str, List[float]], run_name: str) -> None:
    epochs = np.arange(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history['train_loss'], label='Train Loss')
    axes[0].plot(epochs, history['val_loss'], label='Val Loss')
    axes[0].set_title(f'Loss Curves - {run_name}')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='Train Acc')
    axes[1].plot(epochs, history['val_acc'], label='Val Acc')
    axes[1].set_title(f'Accuracy Curves - {run_name}')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()

    out_path = FIGURE_DIR / f'{run_name}_curves.png'
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved training curves: {out_path}')


def collect_misclassified_rows(
    model: nn.Module,
    dataset_subset: Subset,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    class_names: List[str],
) -> pd.DataFrame:
    wrong_idx = np.where(y_true != y_pred)[0]
    rows = []
    for i in wrong_idx:
        dataset_index = dataset_subset.indices[i]
        true_label = int(y_true[i])
        pred_label = int(y_pred[i])
        pred_conf = float(y_prob[i, pred_label])
        rows.append({
            'sample_position': int(i),
            'dataset_index': int(dataset_index),
            'true_idx': true_label,
            'pred_idx': pred_label,
            'true_label': class_names[true_label],
            'pred_label': class_names[pred_label],
            'pred_confidence': pred_conf,
        })

    df = pd.DataFrame(rows).sort_values(by='pred_confidence', ascending=False)
    return df


def denormalize(img_tensor: torch.Tensor) -> np.ndarray:
    img = img_tensor.detach().cpu().numpy().transpose(1, 2, 0)
    mean = np.array(IMAGENET_MEAN)
    std = np.array(IMAGENET_STD)
    img = (img * std) + mean
    return np.clip(img, 0, 1)


def show_misclassified_examples(
    run_name: str,
    dataset_subset: Subset,
    mis_df: pd.DataFrame,
    class_names: List[str],
    max_images: int = 10,
) -> None:
    if mis_df.empty:
        print(f'[{run_name}] No misclassified examples to display.')
        return

    display_df = mis_df.head(max_images)
    n = len(display_df)
    cols = min(5, n)
    rows = int(np.ceil(n / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax_idx, (_, row) in enumerate(display_df.iterrows()):
        ax = axes[ax_idx]
        img_tensor, _ = dataset_subset[row['sample_position']]
        img = denormalize(img_tensor)

        ax.imshow(img)
        ax.axis('off')
        ax.set_title(
            f"T:{row['true_label']} | P:{row['pred_label']}\nConf:{row['pred_confidence']:.2f}",
            fontsize=9
        )

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle(f'Misclassified Examples - {run_name}', y=1.02)
    plt.tight_layout()
    plt.show()


def print_confusion_groups(mis_df: pd.DataFrame) -> None:
    if mis_df.empty:
        print('No confusion groups (no misclassifications).')
        return

    pair_counts = (
        mis_df.groupby(['true_label', 'pred_label'])
        .size()
        .sort_values(ascending=False)
        .reset_index(name='count')
    )
    print('Top confusion types:')
    display(pair_counts.head(10))

    for a, b in [('sad', 'relaxed'), ('happy', 'angry')]:
        subset = mis_df[(mis_df['true_label'] == a) & (mis_df['pred_label'] == b)]
        print(f'{a} -> {b}: {len(subset)}')


def run_experiment(config: ExperimentConfig, dataset_root: Path, run_full: bool = RUN_FULL_EXPERIMENTS) -> RunArtifacts:
    run_name = f"{config.model_name}_{'finetuned' if config.fine_tuned else 'frozen'}_{'aug' if config.augmentation else 'noaug'}"
    print('\n' + '=' * 90)
    print(f'Running: {run_name}')

    loaders, class_names, _ = create_dataloaders(
        dataset_root=dataset_root,
        augmentation=config.augmentation,
        batch_size=config.batch_size,
    )

    model = create_model(config.model_name, num_classes=len(class_names)).to(device)
    model = set_trainable_layers(model, fine_tuned=config.fine_tuned)

    if run_full:
        train_epochs = config.frozen_epochs if not config.fine_tuned else config.frozen_epochs + config.finetune_epochs
    else:
        train_epochs = SMOKE_EPOCHS

    model, history, best_path = train_model(
        model=model,
        loaders=loaders,
        run_name=run_name,
        epochs=train_epochs,
        fine_tuned=config.fine_tuned,
    )

    y_true, y_pred, y_prob = predict_with_probabilities(model, loaders['test'])
    metrics = evaluate_predictions(y_true, y_pred, class_names)

    plot_history(history, run_name)
    plot_confusion(
        y_true,
        y_pred,
        class_names,
        title=f'Confusion Matrix - {run_name}',
        save_name=f'{run_name}_confusion.png',
    )

    mis_df = collect_misclassified_rows(
        model=model,
        dataset_subset=loaders['test'].dataset,
        y_true=y_true,
        y_pred=y_pred,
        y_prob=y_prob,
        class_names=class_names,
    )
    all_misclassified[run_name] = mis_df

    show_misclassified_examples(run_name, loaders['test'].dataset, mis_df, class_names, max_images=10)
    print_confusion_groups(mis_df)

    result_row = {
        'run_name': run_name,
        'model': config.model_name,
        'fine_tuned': config.fine_tuned,
        'augmentation': config.augmentation,
        'accuracy': metrics['accuracy'],
        'precision': metrics['precision_macro'],
        'recall': metrics['recall_macro'],
        'f1_score': metrics['f1_macro'],
        'checkpoint': str(best_path),
    }

    history_by_run[run_name] = history
    results.append(result_row)

    return RunArtifacts(
        run_name=run_name,
        model_name=config.model_name,
        fine_tuned=config.fine_tuned,
        augmentation=config.augmentation,
        best_checkpoint=str(best_path),
        accuracy=metrics['accuracy'],
        precision_macro=metrics['precision_macro'],
        recall_macro=metrics['recall_macro'],
        f1_macro=metrics['f1_macro'],
    )


def run_experiment_matrix(dataset_root: Path, run_full: bool = RUN_FULL_EXPERIMENTS) -> pd.DataFrame:
    experiment_configs = [
        # Experiment 1: Model comparison
        ExperimentConfig(model_name='resnet18', fine_tuned=False, augmentation=False),
        ExperimentConfig(model_name='resnet18', fine_tuned=True, augmentation=False),
        ExperimentConfig(model_name='resnet50', fine_tuned=False, augmentation=False),
        ExperimentConfig(model_name='resnet50', fine_tuned=True, augmentation=False),

        # Experiment 2: Augmentation study on ResNet-50 fine-tuned
        ExperimentConfig(model_name='resnet50', fine_tuned=True, augmentation=False),
        ExperimentConfig(model_name='resnet50', fine_tuned=True, augmentation=True),
    ]

    run_records = []
    for cfg in experiment_configs:
        artifact = run_experiment(cfg, dataset_root=dataset_root, run_full=run_full)
        run_records.append(asdict(artifact))

    return pd.DataFrame(run_records)

In [ ]:
# --- W&B integration: Colab-friendly login + automatic logging ---
from datetime import datetime
import warnings
import sys

try:
    import wandb
    HAS_WANDB = True
except Exception:
    HAS_WANDB = False
    wandb = None

USE_WANDB = True
WANDB_PROJECT = 'dog-emotion'
WANDB_ENTITY = None  # Optional: set your W&B team/user entity string
WANDB_MODE = os.getenv('WANDB_MODE', 'online')  # set to 'offline' if needed
WANDB_RUN = None
IS_COLAB = 'google.colab' in sys.modules


def _disable_wandb(reason: str) -> None:
    global USE_WANDB
    USE_WANDB = False
    warnings.warn(f"W&B disabled: {reason}")


def _ensure_wandb_login() -> bool:
    if not (USE_WANDB and HAS_WANDB):
        return False

    try:
        if not getattr(wandb.api, 'api_key', None):
            print('W&B login required. Paste your API key when prompted.')
            wandb.login(anonymous='never', relogin=False)
        return True
    except Exception as exc:
        _disable_wandb(f'login failed ({exc})')
        return False


def init_wandb_if_enabled() -> None:
    global WANDB_RUN
    if not (USE_WANDB and HAS_WANDB):
        if USE_WANDB and not HAS_WANDB:
            warnings.warn("W&B logging requested but package not installed. Run: pip install wandb")
        return

    if WANDB_RUN is not None:
        return

    if not _ensure_wandb_login():
        return

    run_name = f"emotion-detection-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    config = {
        'seed': SEED,
        'batch_size': BATCH_SIZE,
        'epochs_frozen': EPOCHS_FROZEN,
        'epochs_finetune': EPOCHS_FINETUNE,
        'lr_frozen': LR_FROZEN,
        'lr_finetune': LR_FINETUNE,
        'weight_decay': WEIGHT_DECAY,
        'dropout': DROPOUT,
        'run_full_experiments': RUN_FULL_EXPERIMENTS,
        'run_smoke_demo': RUN_SMOKE_DEMO,
        'smoke_epochs': SMOKE_EPOCHS,
        'class_names': CLASS_NAMES,
        'device': str(device),
        'is_colab': IS_COLAB,
    }

    try:
        WANDB_RUN = wandb.init(
            project=WANDB_PROJECT,
            entity=WANDB_ENTITY,
            name=run_name,
            config=config,
            mode=WANDB_MODE,
            reinit=True,
        )
        print(f"W&B run initialized: {WANDB_RUN.name}")
    except Exception as exc:
        _disable_wandb(f'init failed ({exc})')


def _wandb_log(data: Dict, step: Optional[int] = None) -> None:
    if USE_WANDB and HAS_WANDB and (wandb.run is not None):
        wandb.log(data, step=step)


def _wandb_log_image(key: str, image_path: Path) -> None:
    if USE_WANDB and HAS_WANDB and (wandb.run is not None) and image_path.exists():
        _wandb_log({key: wandb.Image(str(image_path))})


# Wrap existing functions (keeps notebook clean, no duplicate training implementation)
_base_train_model = train_model
_base_plot_history = plot_history
_base_plot_confusion = plot_confusion
_base_show_misclassified_examples = show_misclassified_examples


def train_model(
    model: nn.Module,
    loaders: Dict[str, DataLoader],
    run_name: str,
    epochs: int,
    fine_tuned: bool,
    ) -> Tuple[nn.Module, Dict[str, List[float]], Path]:
    model, history, best_path = _base_train_model(
        model=model,
        loaders=loaders,
        run_name=run_name,
        epochs=epochs,
        fine_tuned=fine_tuned,
    )

    # Log epoch metrics from history with explicit epoch steps
    num_epochs = len(history.get('train_loss', []))
    for i in range(num_epochs):
        _wandb_log({
            f'{run_name}/train_loss': history['train_loss'][i],
            f'{run_name}/val_loss': history['val_loss'][i],
            f'{run_name}/train_acc': history['train_acc'][i],
            f'{run_name}/val_acc': history['val_acc'][i],
            f'{run_name}/lr': history['lr'][i],
            f'{run_name}/epoch': i + 1,
        }, step=i + 1)

    best_val_acc = max(history['val_acc']) if history['val_acc'] else None
    _wandb_log({
        f'{run_name}/best_val_acc': best_val_acc,
        f'{run_name}/best_checkpoint': str(best_path),
    })

    return model, history, best_path


def plot_history(history: Dict[str, List[float]], run_name: str) -> None:
    _base_plot_history(history, run_name)
    out_path = FIGURE_DIR / f'{run_name}_curves.png'
    _wandb_log_image(f'figures/{run_name}_curves', out_path)


def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, class_names: List[str], title: str, save_name: str) -> None:
    _base_plot_confusion(y_true, y_pred, class_names, title, save_name)
    out_path = FIGURE_DIR / save_name
    _wandb_log_image(f'figures/{save_name}', out_path)


def show_misclassified_examples(
    run_name: str,
    dataset_subset: Subset,
    mis_df: pd.DataFrame,
    class_names: List[str],
    max_images: int = 10,
    ) -> None:
    _base_show_misclassified_examples(run_name, dataset_subset, mis_df, class_names, max_images=max_images)

    # Save the currently rendered figure for W&B logging
    out_path = FIGURE_DIR / f'{run_name}_misclassified.png'
    try:
        fig = plt.gcf()
        if fig is not None and len(fig.axes) > 0:
            fig.savefig(out_path, dpi=150, bbox_inches='tight')
            _wandb_log_image(f'figures/{run_name}_misclassified', out_path)
    except Exception:
        pass


init_wandb_if_enabled()
print(f'W&B enabled: {USE_WANDB and HAS_WANDB}')

## 5) Section: Run a Minimal End-to-End Example
Execute the core function with small sample input and display outputs to confirm the implementation flow.

In [ ]:
dataset_root = resolve_dataset_root(DATASET_ROOT)

# Optional smoke run (disabled by default so "Run All" goes straight to full experiments).
if RUN_SMOKE_DEMO:
    smoke_cfg = ExperimentConfig(
        model_name='resnet18',
        fine_tuned=False,
        augmentation=False,
        frozen_epochs=1,
        finetune_epochs=0,
    )

    smoke_artifact = run_experiment(smoke_cfg, dataset_root=dataset_root, run_full=False)
    print('Smoke run complete:')
    print(smoke_artifact)
else:
    print('Smoke demo skipped (RUN_SMOKE_DEMO=False).')

In [ ]:
# Full goals-required experiment execution.
# Set RUN_FULL_EXPERIMENTS=True in config cell to train for full epochs.
# Keeping False is useful for quick validation of notebook logic.

final_df = run_experiment_matrix(dataset_root=dataset_root, run_full=RUN_FULL_EXPERIMENTS)

# Structured results storage as required
results_df = pd.DataFrame(results)
results_df = results_df.drop_duplicates(subset=['run_name'], keep='last').reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / 'results_summary.csv', index=False)

comparison_table = results_df[['model', 'fine_tuned', 'augmentation', 'accuracy', 'f1_score']].copy()
comparison_table = comparison_table.sort_values(by=['model', 'fine_tuned', 'augmentation']).reset_index(drop=True)

print('Final Comparison Table')
display(comparison_table)
print(f"Saved results to: {OUTPUT_DIR / 'results_summary.csv'}")

# W&B: final summary table + artifacts
if USE_WANDB and HAS_WANDB and (wandb.run is not None):
    wandb.log({'results_table': wandb.Table(dataframe=results_df)})
    wandb.log({'comparison_table': wandb.Table(dataframe=comparison_table)})

    results_csv_path = OUTPUT_DIR / 'results_summary.csv'
    if results_csv_path.exists():
        artifact = wandb.Artifact(name='emotion-detection-results', type='results')
        artifact.add_file(str(results_csv_path))
        wandb.log_artifact(artifact)

    wandb.finish()
    print('W&B run finished.')

## 6) Section: Add Basic Assertions and Debug Output
Add lightweight tests with assert statements and print/log output to validate expected behavior during development.

In [ ]:
# Basic validations
assert isinstance(CLASS_NAMES, list) and len(CLASS_NAMES) == 4, 'Expected 4 emotion classes.'
assert BATCH_SIZE > 0, 'BATCH_SIZE must be positive.'
assert (OUTPUT_DIR.exists() and CHECKPOINT_DIR.exists() and FIGURE_DIR.exists()), 'Output directories missing.'

# Validate expected comparison columns when results exist
expected_cols = {'model', 'fine_tuned', 'augmentation', 'accuracy', 'f1_score'}
if len(results) > 0:
    temp_df = pd.DataFrame(results)
    missing = expected_cols - set(temp_df.columns)
    assert not missing, f'Missing result columns: {missing}'
    print('Result schema check passed.')
else:
    print('No experiment results yet; run the full experiment cell above.')

print('Debug checks complete.')